# Reinforcement Learning–Based Active Learning for Semantic Segmentation

This notebook demonstrates a **reinforcement learning (REINFORCE)–based active learning pipeline**
for semantic segmentation using a U-Net backbone.

Unlike heuristic query strategies (entropy, diversity, etc.), the querying policy is **learned**
to directly optimize downstream segmentation performance.

### Goals
- Run a full RL-based active learning loop
- Analyze learning dynamics of the policy
- Compare against heuristic active learning baselines

### Key characteristics
- Policy-gradient (REINFORCE) training
- State = CNN bottleneck features + uncertainty measures
- Reward = ΔF1-score on validation set

## 1. Imports and Environment Setup

In [ ]:
import sys
sys.path.append("..")

import torch
from torch.utils.data import DataLoader
from pathlib import Path
import yaml
import numpy as np


## 2. Experiment Configuration

In [ ]:
config_path = Path("../experiments/configs/rl_active_learning.yaml")
with open(config_path) as f:
    args = argparse.Namespace(**yaml.safe_load(f))

print(args)

## 3. Dataset Preparation

In [ ]:
from src.datasets.unet_dataset import UNetDataset

train_dataset = UNetDataset(args.data_root, is_train=True)
val_dataset   = UNetDataset(args.data_root, is_train=False)

print(f"Train size: {len(train_dataset)}")
print(f"Val size:   {len(val_dataset)}")


## 4. RL Active Learning System Initialization


In [ ]:
from src.rl_active_learning import ActiveLearningSegmentationRL
from src.utils import set_seed

set_seed(args.seed)

device = torch.device(f"cuda:{args.gpu}" if torch.cuda.is_available() else "cpu")
al = ActiveLearningSegmentationRL(train_dataset, args, device)

The system maintains:
- a frozen **oracle model** for feature extraction and uncertainty
- a **main model** trained on the growing labeled set
- a **policy network** trained with REINFORCE to select informative samples


## 5. Cold Start Initialization


In [ ]:
al.diversity_based_initialization(initial_percentage=args.init_pct)
al.train_oracle_model()

We fix the cold-start strategy to diversity sampling in order to isolate
the effect of the reinforcement learning query policy.

## 6. Initial Training Before RL Cycles

In [ ]:
val_loader = DataLoader(
    val_dataset,
    batch_size=args.batch_size,
    shuffle=False,
    num_workers=args.workers,
    pin_memory=True
)

al.train_main_model()
init_metrics = al.evaluate_main_model(val_loader)
al.prev_f1 = init_metrics["f1"]

history = [init_metrics]


## 7. Reinforcement Learning Active Learning Cycles


In [ ]:
for cycle in range(args.al_cycles):
    print(f"\n=== RL-AL Cycle {cycle + 1}/{args.al_cycles} ===")
    print(f"Labeled: {len(al.labeled_indices)}")

    metrics = al.run_cycle(val_loader)
    if metrics is None:
        break

    history.append(metrics)


At each cycle:
1. The policy samples a batch of unlabeled candidates
2. The main model is retrained
3. Validation F1 improvement defines the reward
4. The policy is updated via REINFORCE


## 8. Results and Learning Dynamics

In [ ]:
f1_scores = [h["f1"] for h in history]

plt.plot(f1_scores, "-o")
plt.xlabel("AL Cycle")
plt.ylabel("Validation F1")
plt.title("RL Active Learning Performance")
plt.grid(True)
plt.show()


## 9. Saving Results

In [ ]:
from src.utils import save_results_json

out = {
    "experiment_type": "rl_active_learning",
    "dataset": args.dataset_name,
    "strategy": "rl_policy",
    "seed": args.seed,
    "num_labeled": len(al.labeled_indices),
    "metrics": {
        "f1": {
            "history": [h["f1"] for h in history],
            "final": history[-1]["f1"],
            "best": max(h["f1"] for h in history),
        },
        "iou": {
            "history": [h["iou"] for h in history],
            "final": history[-1]["iou"],
            "best": max(h["iou"] for h in history),
        },
    }
}

save_results_json(out, experiment_name="rl_active_learning_deepcrack")
